# S3-Forecaster y S3-FastSketch ? ICMD Monthly

Notebook reproducible para evaluar **S3Forecaster**, **S3FastSketchForecaster** y los baselines del repositorio S3Forecaster-S3FastSketch sobre la base ICMD almacenada en series_prioritarias.json.

## Decisiones metodol?gicas

- Cada entrada del JSON representa una serie mensual con campos index y values.
- Se usan las observaciones originales; no se crean ni se recortan series artificialmente.
- La evaluaci?n sigue una partici?n temporal **64 % / 16 % / 20 %** para entrenamiento, calibraci?n y prueba.
- Las 24 observaciones de ICMD producen bloques de **15 / 3 / 6**, compatibles con los m?nimos internos del protocolo.
- El HPO utiliza un grupo reproducible de N_HPO_SERIES series (10 por defecto).
- Cada trial puntual se punt?a con la **mediana del SMAPE** entre las series del grupo.
- La optimizaci?n UQ conserva su objetivo propio: mediana de MSIS m?s penalizaci?n por cobertura insuficiente.
- Las series usadas para HPO se excluyen de la evaluaci?n final para evitar fuga de informaci?n.
- No se aplica transformaci?n logar?tmica: el conjunto contiene ceros y al menos un valor negativo.


In [ ]:
# ============================================================
# 1. ENTORNO COMPATIBLE PARA KAGGLE
# ============================================================
# Esta celda evita reinstalar NumPy, SciPy, pandas, scikit-learn
# o PyTorch. Ejecútela antes de importar Transformers/Chronos.

from __future__ import annotations

import subprocess
import sys

INSTALL_RUNTIME = True

RUNTIME_PACKAGES = [
    "huggingface-hub==0.36.0",
    "hf-xet==1.1.10",
    "tokenizers==0.22.1",
    "transformers==4.57.1",
    "safetensors>=0.5.3,<1",
    "accelerate>=1.1,<2",
    "einops>=0.7,<1",
    "chronos-forecasting==2.3.1",
    "optuna>=3.6,<5",
    "statsmodels>=0.14,<1",
    "openpyxl>=3.1,<4",
    "pyarrow>=15,<24",
]

if INSTALL_RUNTIME:
    command = [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--upgrade",
        "--no-cache-dir",
        *RUNTIME_PACKAGES,
    ]
    completed = subprocess.run(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    print(completed.stdout[-12000:])
    if completed.returncode != 0:
        raise RuntimeError(
            "La instalación del entorno falló. Revise la salida mostrada."
        )
else:
    print("INSTALL_RUNTIME=False: se conserva el entorno existente.")

In [ ]:
# ============================================================
# 2. CLONAR E INSTALAR EL REPOSITORIO
# ============================================================

from pathlib import Path
import os
import subprocess
import sys

REPOSITORY_URL = "https://github.com/AlejoPatigno/S3Forecaster-S3FastSketch.git"
REPOSITORY_REF = os.environ.get("S3_REPOSITORY_REF", "main")
WORKING_DIR = Path("/kaggle/working").resolve()
REPOSITORY_DIR = WORKING_DIR / "S3Forecaster-S3FastSketch"

WORKING_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(WORKING_DIR)

if not REPOSITORY_DIR.exists():
    clone_url = REPOSITORY_URL
    github_token = os.environ.get("GITHUB_TOKEN")
    if github_token:
        clone_url = REPOSITORY_URL.replace(
            "https://",
            f"https://x-access-token:{github_token}@",
            1,
        )
    subprocess.run(
        ["git", "clone", "--quiet", clone_url, str(REPOSITORY_DIR)],
        check=True,
    )

subprocess.run(
    ["git", "fetch", "--quiet", "origin", REPOSITORY_REF],
    cwd=REPOSITORY_DIR,
    check=True,
)
subprocess.run(
    ["git", "checkout", "--quiet", REPOSITORY_REF],
    cwd=REPOSITORY_DIR,
    check=True,
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", "-e", str(REPOSITORY_DIR)],
    check=True,
)

if str(REPOSITORY_DIR) not in sys.path:
    sys.path.insert(0, str(REPOSITORY_DIR))

GIT_COMMIT = subprocess.check_output(
    ["git", "rev-parse", "HEAD"],
    cwd=REPOSITORY_DIR,
    text=True,
).strip()

print("Repositorio:", REPOSITORY_DIR)
print("Commit:", GIT_COMMIT)

In [ ]:
# ============================================================
# 3. CONFIGURACI?N DEL EXPERIMENTO
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import os

from s3paper.kaggle_workflow import NotebookRunConfig

SEED = 42
SEASONAL_PERIOD = 12
DATASET_NAME = "ICMD_Monthly"

# La evaluaci?n de cada trial ahora cuesta N_HPO_SERIES ajustes.
# Por eso se reduce el presupuesto respecto del notebook monoserie.
EXPERIMENT_MODE = "standard"

TRIALS_BY_MODE = {
    "smoke": (3, 2),
    "standard": (30, 15),
    "full": (60, 30),
}
POINT_TRIALS, UQ_TRIALS = TRIALS_BY_MODE[EXPERIMENT_MODE]

N_HPO_SERIES = 10
POINT_OBJECTIVE_METRIC = "smape_percent"

TARGET_COVERAGE = 0.90
TRAIN_RATIO = 0.64
CALIBRATION_RATIO = 0.16
TEST_RATIO = 0.20

# Puede fijarse una lista expl?cita de IDs. Con None se seleccionan
# N_HPO_SERIES de forma determinista antes de observar m?tricas.
HPO_SERIES_IDS = None

USE_CHRONOS = True
REQUIRE_CHRONOS = False

PRIOR_NAMES = [
    "causal_rolling_mean",
    "seasonal_naive",
    "ets",
    "theta",
]
if USE_CHRONOS:
    PRIOR_NAMES.append("chronos")

CONFIG = NotebookRunConfig(
    run_mode=EXPERIMENT_MODE,
    seed=SEED,
    development_fraction=0.30,
    n_folds=3,
    validation_size=3,
    point_trials=POINT_TRIALS,
    uq_trials=UQ_TRIALS,
    target_coverage=TARGET_COVERAGE,
    prior_names=tuple(PRIOR_NAMES),
)

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
OUTPUT_DIRECTORY = Path("/kaggle/working/outputs") / DATASET_NAME / RUN_ID
OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=False)

print(CONFIG)
print("N?mero de series HPO:", N_HPO_SERIES)
print("Objetivo puntual:", POINT_OBJECTIVE_METRIC)
print("Salida:", OUTPUT_DIRECTORY)


In [ ]:
# ============================================================
# 4. LOCALIZACIÓN, CARGA Y VALIDACIÓN DEL JSON ICMD
# ============================================================

from __future__ import annotations

from pathlib import Path
from typing import Any
import json
import os
import shutil

import numpy as np
import pandas as pd


def locate_icmd_json(
    filename: str = "series_prioritarias.json",
) -> Path:
    """Localiza el JSON en Kaggle o en el directorio de trabajo."""

    explicit = os.environ.get("ICMD_JSON_PATH")
    if explicit:
        path = Path(explicit).expanduser().resolve()
        if not path.is_file():
            raise FileNotFoundError(f"ICMD_JSON_PATH no existe: {path}")
        return path

    direct_candidates = [
        Path("/kaggle/working") / filename,
        Path.cwd() / filename,
    ]
    for path in direct_candidates:
        if path.is_file():
            return path.resolve()

    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        matches = sorted(kaggle_input.rglob(filename))
        if matches:
            return matches[0].resolve()

        json_candidates = sorted(kaggle_input.rglob("*.json"))
        compatible = [
            path for path in json_candidates
            if "serie" in path.name.lower() or "icmd" in str(path).lower()
        ]
        if compatible:
            return compatible[0].resolve()

    raise FileNotFoundError(
        f"No se encontró {filename}. Adjunte el JSON como Kaggle Dataset "
        "o defina la variable de entorno ICMD_JSON_PATH."
    )


def load_icmd_json(
    json_path: str | Path,
    *,
    expected_frequency: str = "MS",
    minimum_length: int = 18,
) -> tuple[dict[str, pd.Series], pd.DataFrame]:
    """
    Lee el formato:
        {
          "series_id": {
            "index": [...],
            "values": [...]
          }
        }

    La función no imputa, no recorta y no agrega fechas duplicadas.
    """

    json_path = Path(json_path)
    payload: Any = json.loads(json_path.read_text(encoding="utf-8"))

    if not isinstance(payload, dict) or not payload:
        raise ValueError("El JSON debe ser un diccionario no vacío por serie.")

    series_map: dict[str, pd.Series] = {}
    metadata_rows: list[dict[str, Any]] = []

    for raw_series_id, record in payload.items():
        series_id = str(raw_series_id)

        if not isinstance(record, dict):
            raise TypeError(f"{series_id}: el registro debe ser un diccionario.")
        if "index" not in record or "values" not in record:
            raise KeyError(f"{series_id}: faltan 'index' o 'values'.")

        raw_index = record["index"]
        raw_values = record["values"]

        if len(raw_index) != len(raw_values):
            raise ValueError(
                f"{series_id}: index={len(raw_index)} y values={len(raw_values)}."
            )

        index = pd.to_datetime(raw_index, errors="raise")
        if index.has_duplicates:
            duplicates = index[index.duplicated()].astype(str).tolist()
            raise ValueError(f"{series_id}: fechas duplicadas: {duplicates}")

        values = pd.to_numeric(
            pd.Series(raw_values),
            errors="coerce",
        ).to_numpy(dtype=float)

        if not np.isfinite(values).all():
            positions = np.flatnonzero(~np.isfinite(values)).tolist()
            raise ValueError(
                f"{series_id}: valores no finitos en posiciones {positions}."
            )

        order = np.argsort(index)
        index = pd.DatetimeIndex(index[order])
        values = values[order]

        series = pd.Series(
            values,
            index=index,
            name=series_id,
            dtype=float,
        )

        if len(series) < minimum_length:
            raise ValueError(
                f"{series_id}: longitud {len(series)} < mínimo {minimum_length}."
            )

        expected_index = pd.date_range(
            start=series.index.min(),
            periods=len(series),
            freq=expected_frequency,
        )
        if not series.index.equals(expected_index):
            missing = expected_index.difference(series.index)
            extra = series.index.difference(expected_index)
            raise ValueError(
                f"{series_id}: calendario mensual irregular. "
                f"Faltantes={missing.astype(str).tolist()}, "
                f"adicionales={extra.astype(str).tolist()}."
            )

        series_map[series_id] = series

        metadata_rows.append(
            {
                "series_id": series_id,
                "start": series.index.min(),
                "end": series.index.max(),
                "length": len(series),
                "frequency": expected_frequency,
                "n_zero": int((series == 0).sum()),
                "n_negative": int((series < 0).sum()),
                "minimum": float(series.min()),
                "maximum": float(series.max()),
                "mean": float(series.mean()),
                "std": float(series.std(ddof=1)),
            }
        )

    metadata = (
        pd.DataFrame(metadata_rows)
        .sort_values("series_id")
        .reset_index(drop=True)
    )

    return series_map, metadata


ICMD_JSON_PATH = locate_icmd_json()
SERIES_MAP, ICMD_METADATA = load_icmd_json(ICMD_JSON_PATH)

shutil.copy2(
    ICMD_JSON_PATH,
    OUTPUT_DIRECTORY / "series_prioritarias.json",
)

print("Archivo:", ICMD_JSON_PATH)
print("Número de series:", len(SERIES_MAP))
display(ICMD_METADATA)

In [ ]:
# ============================================================
# 5. AUDITORÍA DEL CONJUNTO DE DATOS
# ============================================================

import numpy as np
import pandas as pd

lengths = ICMD_METADATA["length"].to_numpy()
if not np.all(lengths == lengths[0]):
    print("Advertencia: las series tienen longitudes diferentes.")

split_rows = []
for series_id, series in SERIES_MAP.items():
    n = len(series)
    n_train = int(np.floor(n * TRAIN_RATIO))
    n_calibration = int(np.floor(n * CALIBRATION_RATIO))
    n_test = n - n_train - n_calibration

    split_rows.append(
        {
            "series_id": series_id,
            "n_total": n,
            "n_train": n_train,
            "n_calibration": n_calibration,
            "n_test": n_test,
        }
    )

SPLIT_AUDIT = pd.DataFrame(split_rows)

if (SPLIT_AUDIT["n_train"] < 12).any():
    raise ValueError("Alguna serie no alcanza 12 observaciones de entrenamiento.")
if (SPLIT_AUDIT["n_calibration"] < 3).any():
    raise ValueError("Alguna serie no alcanza 3 observaciones de calibración.")
if (SPLIT_AUDIT["n_test"] < 3).any():
    raise ValueError("Alguna serie no alcanza 3 observaciones de prueba.")

DATASET_AUDIT = pd.DataFrame(
    {
        "dataset": [DATASET_NAME],
        "n_series": [len(SERIES_MAP)],
        "minimum_length": [int(ICMD_METADATA["length"].min())],
        "maximum_length": [int(ICMD_METADATA["length"].max())],
        "total_zeros": [int(ICMD_METADATA["n_zero"].sum())],
        "total_negatives": [int(ICMD_METADATA["n_negative"].sum())],
        "seasonal_period": [SEASONAL_PERIOD],
        "train_ratio": [TRAIN_RATIO],
        "calibration_ratio": [CALIBRATION_RATIO],
        "test_ratio": [TEST_RATIO],
    }
)

display(DATASET_AUDIT)
display(SPLIT_AUDIT)

ICMD_METADATA.to_csv(
    OUTPUT_DIRECTORY / "dataset_metadata.csv",
    index=False,
)
SPLIT_AUDIT.to_csv(
    OUTPUT_DIRECTORY / "split_audit.csv",
    index=False,
)
DATASET_AUDIT.to_csv(
    OUTPUT_DIRECTORY / "dataset_audit.csv",
    index=False,
)

In [ ]:
# ============================================================
# 6. VISUALIZACIÓN DESCRIPTIVA
# ============================================================

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

standardized = pd.DataFrame(
    {
        series_id: (
            (series - series.mean()) / series.std(ddof=1)
            if series.std(ddof=1) > 0
            else series * 0.0
        )
        for series_id, series in SERIES_MAP.items()
    }
)

fig, ax = plt.subplots(figsize=(14, 7))
standardized.plot(ax=ax, linewidth=1.5)
ax.axvline(
    standardized.index[int(np.floor(len(standardized) * TRAIN_RATIO))],
    linestyle="--",
    linewidth=1,
    label="Fin aproximado de entrenamiento",
)
ax.set_xlabel("Fecha")
ax.set_ylabel("Valor estandarizado")
ax.set_title("ICMD: trayectorias mensuales estandarizadas")
ax.legend(
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
    fontsize=8,
)
fig.tight_layout()
fig.savefig(
    OUTPUT_DIRECTORY / "icmd_standardized_series.png",
    dpi=220,
    bbox_inches="tight",
)
plt.show()

In [ ]:
# ============================================================
# 7. COMPROBACIÓN OPCIONAL DE CHRONOS
# ============================================================

import numpy as np

EFFECTIVE_PRIOR_NAMES = list(CONFIG.prior_names)
CHRONOS_AVAILABLE = False

if "chronos" in EFFECTIVE_PRIOR_NAMES:
    try:
        from s3paper.chronos import (
            ChronosZeroShotModel,
            chronos_predict_fixed_horizon,
        )

        smoke_series = next(iter(SERIES_MAP.values()))
        chronos_model = ChronosZeroShotModel(
            model_id="amazon/chronos-bolt-small",
            device_map="cpu",
        )
        chronos_smoke_forecast = chronos_predict_fixed_horizon(
            chronos_model,
            smoke_series.iloc[:18],
            horizon=1,
        )

        if not np.isfinite(
            chronos_smoke_forecast.to_numpy(dtype=float)
        ).all():
            raise RuntimeError("Chronos devolvió valores no finitos.")

        CHRONOS_AVAILABLE = True
        print("Chronos disponible:", chronos_smoke_forecast.to_dict())

    except Exception as exc:
        print(
            "Chronos no está disponible en este entorno:",
            f"{type(exc).__name__}: {exc}",
        )
        if REQUIRE_CHRONOS:
            raise

        EFFECTIVE_PRIOR_NAMES = [
            name for name in EFFECTIVE_PRIOR_NAMES if name != "chronos"
        ]
        print(
            "Se continuará sin Chronos. Priors efectivos:",
            EFFECTIVE_PRIOR_NAMES,
        )
else:
    print("Chronos fue desactivado mediante USE_CHRONOS=False.")

In [ ]:
# ============================================================
# 8. SELECCI?N REPRODUCIBLE DEL GRUPO DE HPO
# ============================================================

import hashlib


def deterministic_series_order(
    series_ids,
    *,
    seed: int,
):
    def score(series_id: str) -> str:
        return hashlib.sha256(
            f"{seed}:{series_id}".encode("utf-8")
        ).hexdigest()

    return sorted(map(str, series_ids), key=score)


eligible_hpo_ids = [
    str(series_id)
    for series_id, series in SERIES_MAP.items()
    if len(series) >= 18
]

if HPO_SERIES_IDS is None:
    if N_HPO_SERIES <= 0:
        raise ValueError("N_HPO_SERIES debe ser positivo.")
    if N_HPO_SERIES >= len(eligible_hpo_ids):
        raise ValueError(
            "N_HPO_SERIES debe dejar al menos una serie para evaluaci?n."
        )
    HPO_SERIES_IDS = deterministic_series_order(
        eligible_hpo_ids,
        seed=SEED,
    )[:N_HPO_SERIES]
else:
    HPO_SERIES_IDS = [str(series_id) for series_id in HPO_SERIES_IDS]
    if len(HPO_SERIES_IDS) != len(set(HPO_SERIES_IDS)):
        raise ValueError("HPO_SERIES_IDS contiene IDs duplicados.")
    missing = [
        series_id
        for series_id in HPO_SERIES_IDS
        if series_id not in SERIES_MAP
    ]
    if missing:
        raise KeyError(f"Series HPO inexistentes: {missing}")

EVALUATION_IDS = sorted(
    str(series_id)
    for series_id in SERIES_MAP
    if str(series_id) not in set(HPO_SERIES_IDS)
)

print("Series de HPO:", HPO_SERIES_IDS)
print("Series de evaluaci?n:", EVALUATION_IDS)

pd.DataFrame(
    {
        "series_id": HPO_SERIES_IDS,
        "selection_seed": SEED,
        "selection_rule": "sha256(seed:series_id)",
    }
).to_csv(
    OUTPUT_DIRECTORY / "hpo_series.csv",
    index=False,
)

pd.DataFrame(
    {"series_id": EVALUATION_IDS}
).to_csv(
    OUTPUT_DIRECTORY / "evaluation_series.csv",
    index=False,
)


In [ ]:
# ============================================================
# 9. HPO EN UN GRUPO Y TRANSFERENCIA A SERIES NO VISTAS
# ============================================================

from s3paper.single_series_transfer_hpo import (
    run_group_hpo_transfer_experiment,
)

COMMON_HPO_ARGUMENTS = {
    "series_map": SERIES_MAP,
    "dataset_name": DATASET_NAME,
    "seasonal_period": SEASONAL_PERIOD,
    "n_hpo_series": N_HPO_SERIES,
    "hpo_series_ids": HPO_SERIES_IDS,
    "seed": SEED,
    "n_point_trials": CONFIG.point_trials,
    "n_uq_trials": CONFIG.uq_trials,
    "target_coverage": CONFIG.target_coverage,
    "train_ratio": TRAIN_RATIO,
    "calibration_ratio": CALIBRATION_RATIO,
    "test_ratio": TEST_RATIO,
    "point_objective_metric": POINT_OBJECTIVE_METRIC,
    "git_commit": GIT_COMMIT,
    "prior_names": EFFECTIVE_PRIOR_NAMES,
}

S3_RESULT = run_group_hpo_transfer_experiment(
    **COMMON_HPO_ARGUMENTS,
    model_name="S3Forecaster",
    output_dir=OUTPUT_DIRECTORY / "s3_forecaster",
)

FASTSKETCH_RESULT = run_group_hpo_transfer_experiment(
    **COMMON_HPO_ARGUMENTS,
    model_name="S3FastSketchForecaster",
    output_dir=OUTPUT_DIRECTORY / "s3_fastsketch",
)

RESULTS = {
    "config": CONFIG,
    "output_directory": OUTPUT_DIRECTORY,
    "development_ids": HPO_SERIES_IDS,
    "evaluation_ids": EVALUATION_IDS,
    "s3_result": S3_RESULT,
    "fastsketch_result": FASTSKETCH_RESULT,
}

print("S3 best point params:", S3_RESULT["best_point_params"])
print("S3 best UQ params:", S3_RESULT["best_uq_params"])
print("FastSketch best point params:", FASTSKETCH_RESULT["best_point_params"])
print("FastSketch best UQ params:", FASTSKETCH_RESULT["best_uq_params"])


In [ ]:
# ============================================================
# 10. BASELINES CON EL MISMO PROTOCOLO TEMPORAL Y CONFORMAL
# ============================================================

from __future__ import annotations

import time
import pandas as pd

from s3paper.baselines import ConformalizedBaseline
from s3paper.kaggle_workflow import default_baseline_parameters
from s3paper.metrics import seasonal_naive_scale
from s3paper.rolling_protocol import evaluate_rolling_model
from s3paper.single_series_transfer_hpo import temporal_train_cal_test_split


def icmd_baseline_parameters(
    seasonal_period: int = 12,
) -> dict[str, dict]:
    """
    Adapta únicamente las ventanas que son incompatibles con
    una historia efectiva de 18 puntos.

    No se modifica la serie ni el horizonte.
    """

    parameters = default_baseline_parameters(seasonal_period)

    # Con 18 puntos de historia, la inicialización estacional ETS
    # y la desestacionalización Theta pueden requerir dos ciclos.
    parameters["ETS"]["seasonal"] = None
    parameters["ETS"].pop("seasonal_periods", None)
    parameters["Theta"]["deseasonalize"] = False

    short_window = min(6, seasonal_period)
    parameters["AR"]["lags"] = short_window
    parameters["KRR"]["input_window"] = short_window
    parameters["GPR"]["input_window"] = short_window
    parameters["NLinear"]["window_size"] = short_window
    parameters["DLinear"]["window_size"] = short_window
    parameters["DLinear"]["kernel_size"] = 3

    return parameters


def evaluate_baselines_transfer_protocol(
    series_map,
    evaluation_ids,
    *,
    baseline_parameters,
    seasonal_period=12,
    target_coverage=0.90,
    train_ratio=0.64,
    calibration_ratio=0.16,
    test_ratio=0.20,
    aci_step_size=0.05,
    interval_scale=1.0,
    minimum_width_factor=0.0,
):
    alpha = 1.0 - float(target_coverage)

    rows = []
    forecasts = []
    failures = []

    for series_id in evaluation_ids:
        series_id = str(series_id)
        raw_series = series_map[series_id]

        try:
            train, calibration, test = temporal_train_cal_test_split(
                raw_series,
                train_ratio=train_ratio,
                calibration_ratio=calibration_ratio,
                test_ratio=test_ratio,
            )

            fit_history = pd.concat([train, calibration])

            minimum_width = minimum_width_factor * seasonal_naive_scale(
                fit_history,
                seasonal_period=seasonal_period,
            )

            for model_name, params in baseline_parameters.items():
                started = time.perf_counter()

                try:
                    model = ConformalizedBaseline(
                        model_name,
                        dict(params),
                        alpha=alpha,
                        aci_step_size=aci_step_size,
                        interval_scale=interval_scale,
                        minimum_width=minimum_width,
                    )

                    result = evaluate_rolling_model(
                        model,
                        fit_history,
                        test,
                        alpha=alpha,
                        seasonal_period=seasonal_period,
                    )

                    elapsed = time.perf_counter() - started

                    rows.append(
                        {
                            "series_id": series_id,
                            "model": model_name,
                            **result["metrics"],
                            "elapsed_seconds": elapsed,
                            "status": "ok",
                            "error": "",
                        }
                    )

                    forecast = result["forecast"].copy()
                    forecast["series_id"] = series_id
                    forecast["model"] = model_name
                    forecasts.append(forecast)

                except Exception as exc:
                    elapsed = time.perf_counter() - started
                    failure = {
                        "series_id": series_id,
                        "model": model_name,
                        "elapsed_seconds": elapsed,
                        "status": "failed",
                        "error": f"{type(exc).__name__}: {exc}",
                    }
                    rows.append(failure)
                    failures.append(failure)

        except Exception as exc:
            failures.append(
                {
                    "series_id": series_id,
                    "model": "ALL_BASELINES",
                    "status": "failed",
                    "error": f"{type(exc).__name__}: {exc}",
                }
            )

    return {
        "per_series_metrics": pd.DataFrame(rows),
        "forecasts": (
            pd.concat(forecasts, ignore_index=False)
            if forecasts else pd.DataFrame()
        ),
        "failed_series": pd.DataFrame(failures),
        "parameters": baseline_parameters,
    }


BASELINE_PARAMETERS = icmd_baseline_parameters(SEASONAL_PERIOD)

BASELINE_RESULTS = evaluate_baselines_transfer_protocol(
    series_map=SERIES_MAP,
    evaluation_ids=EVALUATION_IDS,
    baseline_parameters=BASELINE_PARAMETERS,
    seasonal_period=SEASONAL_PERIOD,
    target_coverage=CONFIG.target_coverage,
    train_ratio=TRAIN_RATIO,
    calibration_ratio=CALIBRATION_RATIO,
    test_ratio=TEST_RATIO,
)

display(BASELINE_RESULTS["failed_series"])

In [ ]:
# ============================================================
# 11. CONSOLIDACIÓN, COHORTE COMÚN Y TABLAS DEL ARTÍCULO
# ============================================================

from __future__ import annotations

from typing import Any
import numpy as np
import pandas as pd


def extract_model_metrics(
    result: dict[str, Any],
    model_name: str,
) -> pd.DataFrame:
    frame = result["evaluation_results"]["per_series_metrics"].copy()
    frame["model"] = model_name
    frame["status"] = "ok"
    frame["error"] = ""
    return frame


proposed_frames = [
    extract_model_metrics(S3_RESULT, "S3Forecaster"),
    extract_model_metrics(
        FASTSKETCH_RESULT,
        "S3FastSketchForecaster",
    ),
]

baseline_metrics = BASELINE_RESULTS["per_series_metrics"].copy()

ALL_PER_SERIES = pd.concat(
    proposed_frames + [baseline_metrics],
    ignore_index=True,
    sort=False,
)

ALL_PER_SERIES["series_id"] = ALL_PER_SERIES["series_id"].astype(str)
ALL_PER_SERIES["model"] = ALL_PER_SERIES["model"].astype(str)

successful = ALL_PER_SERIES[
    ALL_PER_SERIES["status"].fillna("ok").eq("ok")
].copy()

MODEL_COVERAGE = (
    ALL_PER_SERIES.groupby("model", dropna=False)
    .agg(
        attempted_series=("series_id", "nunique"),
        successful_series=(
            "status",
            lambda values: int(values.fillna("ok").eq("ok").sum()),
        ),
        failed_rows=(
            "status",
            lambda values: int(values.fillna("ok").ne("ok").sum()),
        ),
    )
    .reset_index()
)
MODEL_COVERAGE["success_rate"] = (
    MODEL_COVERAGE["successful_series"]
    / MODEL_COVERAGE["attempted_series"].clip(lower=1)
)

all_attempted_models = sorted(
    ALL_PER_SERIES["model"].dropna().astype(str).unique()
)

success_sets = {
    model: set(
        successful.loc[
            successful["model"].eq(model),
            "series_id",
        ].astype(str)
    )
    for model in all_attempted_models
}

COMMON_SERIES_IDS = (
    sorted(set.intersection(*success_sets.values()))
    if success_sets else []
)

if not COMMON_SERIES_IDS:
    print(
        "ADVERTENCIA: la cohorte común entre todos los modelos está vacía. "
        "Revise MODEL_COVERAGE y los fallos. No se recortarán series "
        "artificialmente."
    )

COMMON_PER_SERIES = successful[
    successful["series_id"].isin(COMMON_SERIES_IDS)
].copy()


def aggregate_results(
    frame: pd.DataFrame,
) -> pd.DataFrame:
    if frame.empty:
        return pd.DataFrame()

    candidate_metrics = [
        "mape_percent",
        "smape_percent",
        "mase",
        "rmse",
        "ecp",
        "mean_width",
        "msis",
        "trainable_params",
        "elapsed_seconds",
    ]
    metric_columns = [
        metric
        for metric in candidate_metrics
        if metric in frame.columns
    ]

    rows = []
    for model, group in frame.groupby("model", dropna=False):
        row = {
            "model": model,
            "n_series": int(group["series_id"].nunique()),
        }

        for metric in metric_columns:
            values = pd.to_numeric(
                group[metric],
                errors="coerce",
            ).replace([np.inf, -np.inf], np.nan).dropna()

            if values.empty:
                continue

            q1 = float(values.quantile(0.25))
            q3 = float(values.quantile(0.75))

            row[f"{metric}_mean"] = float(values.mean())
            row[f"{metric}_median"] = float(values.median())
            row[f"{metric}_iqr"] = q3 - q1

        rows.append(row)

    result = pd.DataFrame(rows)

    ranking_column = next(
        (
            column
            for column in [
                "mape_percent_median",
                "mase_median",
                "smape_percent_median",
                "rmse_median",
            ]
            if column in result.columns
        ),
        None,
    )

    if ranking_column is not None:
        result = result.sort_values(
            ranking_column,
            ascending=True,
        ).reset_index(drop=True)
        result.insert(
            0,
            "rank",
            np.arange(1, len(result) + 1),
        )

    return result


DESCRIPTIVE_TABLE = aggregate_results(successful)
COMMON_COHORT_TABLE = aggregate_results(COMMON_PER_SERIES)

preferred_columns = [
    "rank",
    "model",
    "n_series",
    "mape_percent_median",
    "mape_percent_iqr",
    "smape_percent_median",
    "smape_percent_iqr",
    "mase_median",
    "mase_iqr",
    "rmse_median",
    "rmse_iqr",
    "ecp_mean",
    "mean_width_median",
    "msis_median",
    "trainable_params_median",
    "elapsed_seconds_median",
]

if not DESCRIPTIVE_TABLE.empty:
    DESCRIPTIVE_TABLE = DESCRIPTIVE_TABLE[
        [column for column in preferred_columns if column in DESCRIPTIVE_TABLE]
    ]

if not COMMON_COHORT_TABLE.empty:
    COMMON_COHORT_TABLE = COMMON_COHORT_TABLE[
        [column for column in preferred_columns if column in COMMON_COHORT_TABLE]
    ]

print("Cobertura por modelo")
display(MODEL_COVERAGE.sort_values("model"))

print("Tabla descriptiva por modelo")
display(DESCRIPTIVE_TABLE)

print("Tabla sobre cohorte común")
display(COMMON_COHORT_TABLE)

In [ ]:
# ============================================================
# 12. EXPORTACIÓN COMPLETA
# ============================================================

from pathlib import Path
import json
import zipfile
import pandas as pd

RESULT_DIRECTORY = OUTPUT_DIRECTORY / "main_comparison"
RESULT_DIRECTORY.mkdir(parents=True, exist_ok=True)

ALL_PER_SERIES.to_csv(
    RESULT_DIRECTORY / "icmd_all_per_series.csv",
    index=False,
)
COMMON_PER_SERIES.to_csv(
    RESULT_DIRECTORY / "icmd_common_cohort_per_series.csv",
    index=False,
)
MODEL_COVERAGE.to_csv(
    RESULT_DIRECTORY / "icmd_model_coverage.csv",
    index=False,
)
DESCRIPTIVE_TABLE.to_csv(
    RESULT_DIRECTORY / "icmd_descriptive_table.csv",
    index=False,
)
COMMON_COHORT_TABLE.to_csv(
    RESULT_DIRECTORY / "icmd_common_cohort_table.csv",
    index=False,
)
BASELINE_RESULTS["failed_series"].to_csv(
    RESULT_DIRECTORY / "icmd_baseline_failures.csv",
    index=False,
)

if not BASELINE_RESULTS["forecasts"].empty:
    BASELINE_RESULTS["forecasts"].to_parquet(
        RESULT_DIRECTORY / "icmd_baseline_forecasts.parquet",
        index=True,
    )

experiment_manifest = {
    "dataset": DATASET_NAME,
    "source_json": str(ICMD_JSON_PATH),
    "repository_commit": GIT_COMMIT,
    "run_id": RUN_ID,
    "seed": SEED,
    "seasonal_period": SEASONAL_PERIOD,
    "hpo_series_ids": HPO_SERIES_IDS,
    "n_hpo_series": len(HPO_SERIES_IDS),
    "point_objective_metric": POINT_OBJECTIVE_METRIC,
    "evaluation_series_ids": EVALUATION_IDS,
    "effective_prior_names": EFFECTIVE_PRIOR_NAMES,
    "chronos_available": CHRONOS_AVAILABLE,
    "point_trials": CONFIG.point_trials,
    "uq_trials": CONFIG.uq_trials,
    "target_coverage": CONFIG.target_coverage,
    "train_ratio": TRAIN_RATIO,
    "calibration_ratio": CALIBRATION_RATIO,
    "test_ratio": TEST_RATIO,
    "common_series_ids": COMMON_SERIES_IDS,
    "no_artificial_trimming": True,
    "log_transform": False,
}

(OUTPUT_DIRECTORY / "experiment_manifest.json").write_text(
    json.dumps(
        experiment_manifest,
        indent=2,
        sort_keys=True,
        default=str,
    ),
    encoding="utf-8",
)

excel_path = RESULT_DIRECTORY / "icmd_main_results.xlsx"
with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    ALL_PER_SERIES.to_excel(
        writer,
        sheet_name="all_per_series",
        index=False,
    )
    COMMON_PER_SERIES.to_excel(
        writer,
        sheet_name="common_per_series",
        index=False,
    )
    MODEL_COVERAGE.to_excel(
        writer,
        sheet_name="model_coverage",
        index=False,
    )
    DESCRIPTIVE_TABLE.to_excel(
        writer,
        sheet_name="descriptive",
        index=False,
    )
    COMMON_COHORT_TABLE.to_excel(
        writer,
        sheet_name="common_cohort",
        index=False,
    )
    BASELINE_RESULTS["failed_series"].to_excel(
        writer,
        sheet_name="failures",
        index=False,
    )
    ICMD_METADATA.to_excel(
        writer,
        sheet_name="metadata",
        index=False,
    )

archive_path = OUTPUT_DIRECTORY / f"{DATASET_NAME}_{RUN_ID}.zip"
with zipfile.ZipFile(
    archive_path,
    mode="w",
    compression=zipfile.ZIP_DEFLATED,
) as archive:
    for path in OUTPUT_DIRECTORY.rglob("*"):
        if path.is_file() and path != archive_path:
            archive.write(
                path,
                arcname=path.relative_to(OUTPUT_DIRECTORY),
            )

print("Excel:", excel_path)
print("Archivo consolidado:", archive_path)
print("Directorio de resultados:", OUTPUT_DIRECTORY)

## Interpretaci?n de las salidas

1. icmd_model_coverage.csv debe revisarse antes de comparar modelos. Un modelo con fallos no debe interpretarse como si hubiera sido evaluado sobre la misma cohorte.
2. icmd_common_cohort_table.csv es la tabla principal cuando contiene una cohorte com?n no vac?a.
3. icmd_descriptive_table.csv conserva todos los resultados exitosos por modelo, pero sus tama?os muestrales pueden diferir.
4. icmd_baseline_failures.csv documenta expl?citamente incompatibilidades o errores de ajuste.
5. Los par?metros puntuales se seleccionan minimizando la mediana del SMAPE sobre N_HPO_SERIES; los par?metros UQ usan la mediana de MSIS m?s penalizaci?n de cobertura.
6. Las series de HPO quedan fuera de la evaluaci?n final y los par?metros permanecen congelados durante la transferencia.
7. Las adaptaciones de los baselines afectan ?nicamente configuraciones inviables con 18 observaciones de historia efectiva; no alteran las series ni crean observaciones.
